<a href="https://colab.research.google.com/github/charlien12/ML-Projects/blob/main/SentimentAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
df=pd.read_csv('IMDB Dataset.csv', engine='python')

In [17]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,1
1,wonder littl product film techniqu unassum old...,1
2,thought wonder way spend time hot summer weeke...,1
3,basic there famili littl boy jake think there ...,0
4,petter mattei love time money visual stun film...,1


In [5]:
df.shape

(50000, 2)

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

# Preprocessing

# 1. Converting Lower Case

In [8]:
df['review']=df['review'].str.lower()

# 2. Removing Urls,Special Character, and Html tags

In [9]:
import re

def clean_text(text):
    text = re.sub(r"<.*?>", "", text)          # remove HTML tags
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text) # remove special characters
    text = re.sub(r"\s+", " ", text)           # remove extra spaces
    return text.strip()

df['review'] = df['review'].apply(clean_text)

# 3. Remove Stop words

In [10]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [11]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [12]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]
    return ' '.join(filtered_tokens)
df['review']=df['review'].apply(remove_stopwords)

# 4.Stemming

In [13]:
from nltk.stem import PorterStemmer

In [14]:
def stemming(text):
    stemmer=PorterStemmer()
    tokens=word_tokenize(text)
    stemmed_tokens=[stemmer.stem(token) for token in tokens]
    return " ".join(stemmed_tokens) # Joined with a space
df['review']=df['review'].apply(stemming)

# 5. Encoding

In [16]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df['sentiment']=le.fit_transform(df['sentiment'])

# 6. Vectorization

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [19]:
tf=TfidfVectorizer(max_features=5000)
x=tf.fit_transform(df['review'])

# Dataset and Dataloaders

In [21]:
from sklearn.model_selection import train_test_split
Y=df['sentiment']
x_train,x_test,y_train,y_test=train_test_split(x,Y,test_size=0.2,random_state=42)

In [24]:
from torch.utils.data import TensorDataset,DataLoader
import torch

In [25]:
x_train=x_train.toarray()
x_test=x_test.toarray()
train_set=TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set=TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [26]:
train_loader=DataLoader(train_set,batch_size=64,shuffle=True)
test_loader=DataLoader(test_set,batch_size=64,shuffle=True)

# Build RNN

In [27]:
import torch.nn as nn
import torch.optim as optimizer

In [28]:
class RNN(nn.Module):
  def __init__(self, input_size, hidden_dim=128, n_layers=1):
    super().__init__()
    self.hidden_dim = hidden_dim
    self.n_layers = n_layers
    self.rnn=nn.RNN(input_size,hidden_dim,n_layers,batch_first=True)
    self.fc=nn.Linear(hidden_dim,1)
  def forward(self,x):
    batch_size=torch.zeros(self.n_layers,x.size(0),self.hidden_dim)
    out,_=self.rnn(x,batch_size)
    out=self.fc(out[:,-1,:])
    return out


In [30]:
input_size=x_train.shape[1]
model=RNN(input_size)

In [31]:
criteria=nn.BCELoss()
optim=optimizer.Adam(model.parameters())

# Train the Model

In [32]:
epochs=20
for epoch in range(epochs):
  model.train()
  for xb,yb in train_loader:
    optim.zero_grad()
    xb=xb.unsqueeze(1)
    outputs=model(xb)
    outputs=torch.sigmoid(outputs.squeeze())
    loss=criteria(outputs,yb)
    loss.backward()
    optim.step()
  print(f"Epoch {epoch+1}/{epochs} Loss: {loss.item()}")

Epoch 1/20 Loss: 0.18770065903663635
Epoch 2/20 Loss: 0.26936736702919006
Epoch 3/20 Loss: 0.4326826333999634
Epoch 4/20 Loss: 0.1369229555130005
Epoch 5/20 Loss: 0.1342039406299591
Epoch 6/20 Loss: 0.19601525366306305
Epoch 7/20 Loss: 0.1570035219192505
Epoch 8/20 Loss: 0.11943095177412033
Epoch 9/20 Loss: 0.23316511511802673
Epoch 10/20 Loss: 0.11902932822704315
Epoch 11/20 Loss: 0.15523208677768707
Epoch 12/20 Loss: 0.21502786874771118
Epoch 13/20 Loss: 0.2128814458847046
Epoch 14/20 Loss: 0.2173294872045517
Epoch 15/20 Loss: 0.09593738615512848
Epoch 16/20 Loss: 0.18113943934440613
Epoch 17/20 Loss: 0.27291905879974365
Epoch 18/20 Loss: 0.2388068288564682
Epoch 19/20 Loss: 0.23664188385009766
Epoch 20/20 Loss: 0.11598063260316849


In [33]:
#evaluate the model
model.eval()
with torch.no_grad():
  correct_vals=0
  total_vals=0
  for xb,yb in test_loader:
    xb=xb.unsqueeze(1)
    outputs=model(xb)
    predicted=(torch.sigmoid(outputs.squeeze())>0.5).float()
    correct_vals+=(predicted==yb).sum().item()
    total_vals+=yb.size(0)
  print(f"accuracy= {correct_vals/total_vals*100}")

accuracy= 86.8407784612282
